## PMR3404 - Controle I
## Experiência 5: Projeto de controladores no domínio da frequência

## Identificação

**Aluno1 :Matheus Alexandrino Brito**           **NUSP:11261419**

**Aluno2 :**           **NUSP:**

**Turma de Laboratório:**

**Professor:**

**Versão 2026**

### Instruções para submissão do relatório

1. Todo relatório deve ser submetido no sistema Moodle de acordo com o deadline estabelecido.
2. Submeta um arquivo compactado contendo o arquivo principal do jupyter notebook (.ipynb) e o folder ./Figuras mesmo que não tenha colocado imagens adicionais.
3. Também no arquivo compactado acrescente uma versão no formato HTML do arquivo jupyter notebook. No menu acesse "File->Save and Export Notebook as ...". O arquivo será utilizado para avaliação do relatório, caso haja dúvidas sobre os resultados o arquivo .ipynb será verificado.
4. **Nos arquivos .ipynb e .html você deve manter as células que contêm as saídas das execuções dos scripts em Python com as últimas respostas obtidas. Seções que não apresentarem indícios que os scripts foram executados não serão consideradas para nota.**

### 1. Critério de estabilidade de Nyquist

<img src="./Figuras/StandardBlock.png" width="60%" height="60%"/>

**O critério de estabilidade de Nyquist permite inferir a condição de estabilidade de malha fechada a partir de informações da malha aberta.**

Em última instância deseja-se saber qual o número de pólos de malha fechada no semi-plano direito (SPD).
A estabilidade requer que o número de pólos de malha fechada no semi-plano direito seja nulo.

O critério pode se resumido da seguinte forma. 
Sabemos que:
$$
Z=N+P,
$$
onde:
- $P$ é o número de pólos de $G(s)H(s)$ no SPD,
- $N$ é número de enlaçamentos do ponto $-1$ no sentido horário se $N>0$, e anti-horário se $N<0$,
- $Z$ é o número de zeros de $1+G(s)H(s)$ (**Obs: esse zeros são obviamente os pólos de malha fechada**).

Para estabilidade devemos então ter sempre $Z=0$:
- Se $P>0$ então devemos ter $N=-P$,
- Se $P=0$ então devemos ter $N=0$.

Seja um sistema de controle em malha fechada onde a malha aberta é definida pela seguinte função de transferência:
$$
G(s)H(s)=\frac{K}{s(s+1)^2}
$$

a-) Utilizando o critério de estabilidade de Routh-Hurwitz calcule o valor de $K$ para um sistema marginalmente estável.

**[resposta]**

b-) Utilizando o critério de estabilidade de Nyquist calcule a condição de estabilidade do sistema para os seguintes valores da constante $K$

1) $K=1$:

Utilize o script abaixo para plotar o gráfico de Nyquist e estime o valor de $N$:

**[insira o gráfico gerado aqui]**

- Qual a sua conclusão sobre a condição de estabilidade ?

**[resposta]**

2) $K=2$:

Utilize o script abaixo para plotar o gráfico de Nyquist e estime o valor de $N$:

**[insira o gráfico gerado aqui]**

- Qual a sua conclusão sobre a condição de estabilidade ?

**[resposta]**

3) $K=3$:

Utilize o script abaixo para plotar o gráfico de Nyquist e estime o valor de $N$:

**[insira o gráfico gerado aqui]**

- Qual a sua conclusão sobre a condição de estabilidade ?

**[resposta]**

## Importa os pacotes fundamentais

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control.matlab as ml
import control.matlab as co

## Gráfico de Nyquist com Escala Logarítmica

A função ``closed_log_nyquist()`` plota o gráfico de Nyquist com uma escala em que as porções do gráfico que se encontram no infinito aparece numa área delimitada. Dessa forma é mais fácil deduzir o número de enlaçamentos $N$.

In [ ]:
def cln_transform(ny_data, n=2):
    mag = np.abs(ny_data)
    phase = np.angle(ny_data)
    exp_val = np.log10(n)
    
    # Apply piecewise transformation
    res_mag = np.where(mag < 1, 
                       mag**exp_val, 
                       2 - (mag**(-exp_val)))
    
    return res_mag * np.exp(1j * phase)
def plot_cln_grid(n=2, nd=4):
    theta = np.linspace(0, 2*np.pi, 500)
    # Unit circle (0 dB)
    plt.plot(np.cos(theta), np.sin(theta), 'k-', lw=1.5, label='0 dB')
    # Outer boundary (Infinity)
    plt.plot(2*np.cos(theta), 2*np.sin(theta), 'k-', lw=1)
    
    # Level lines
    for i in range(1, nd + 1):
        r_inner = 1 / (n**i)
        r_outer = 2 - (1 / (n**i))
        plt.plot(r_inner * np.cos(theta), r_inner * np.sin(theta), 'k:', alpha=0.5)
        plt.plot(r_outer * np.cos(theta), r_outer * np.sin(theta), 'k:', alpha=0.5)
        
    # Radial lines (every 30 degrees)
    for deg in range(0, 360, 30):
        rad = np.deg2rad(deg)
        plt.plot([0, 2*np.cos(rad)], [0, 2*np.sin(rad)], 'k:', alpha=0.3)
def get_origin_arc(sys, start_val, n=2):
    # Count poles at s=0
    poles = ml.pole(sys)
    n_poles_origin = np.sum(np.isclose(poles, 0))
    if n_poles_origin == 0:
        return None
    
    # The arc goes from -phase to +phase (or vice versa)
    start_angle = np.angle(np.conj(start_val))
    angles = np.linspace(start_angle, start_angle - n_poles_origin * np.pi, 100)
    return 2 * np.exp(1j * angles)
def closed_log_nyquist(sys, n=2):
    # 1. Frequency Response
    w = np.logspace(-4, 4, 2000)
    mag, phase, omega = ml.freqresp(sys, w)
    ny_data = np.squeeze(mag * np.exp(1j * phase))
    
    # 2. Transform Data
    ny_transformed = cln_transform(ny_data, n)
    
    # 3. Plotting
    plt.figure(figsize=(8, 8))
    plot_cln_grid(n)
    
    # Positive frequencies (Solid)
    plt.plot(ny_transformed.real, ny_transformed.imag, 'r-', lw=2, label='w > 0')
    # Negative frequencies (Dashed)
    plt.plot(ny_transformed.real, -ny_transformed.imag, 'r--', lw=1, label='w < 0')
    
    # 4. Handle closure for poles at origin
    arc = get_origin_arc(sys, ny_transformed[0], n)
    if arc is not None:
        plt.plot(arc.real, arc.imag, 'r--')
    
    # Critical Point (-1, 0)
    plt.plot(-1, 0, 'rx', markersize=10, mew=2)
    plt.text(-1.2, 0.1, '-1', color='red', fontweight='bold')
    
    plt.axis('equal')
    plt.title('Closed Logarithmic Nyquist Plot')
    plt.grid(False)
    plt.show()

In [ ]:
%matplotlib inline
#%matplotlib qt      
s=co.tf('s')
K=1
GH=K/(s*(s+1)**2)
#[a,b,omega]=co.nyquist(GH,omega)
closed_log_nyquist(GH) 

### Exercício 2: Projeto com controlador avanço de fase

O controlador por avanço de fase pode ser escrito através da seguinte equação:
$$
G_c(s) = K_c \alpha\frac{Ts+1}{\alpha Ts+1} = K_c \frac{s+\frac{1}{T}}{s+\frac{1}{\alpha T}}
$$

Um diagrama de Bode para $K_c=1$ e $\alpha=0.1$ é ilustrado abaixo:

<img src="./Figuras/leadcontrol.png" width="50%" height="50%"/>

Seja o seguinte sistema:
$$
G(s) = \frac{4}{s(\frac{5}{2}s+1)(\frac{1}{6}s+1)}
$$

O sistema de controle em malha fechada correspondente é ilustrado na figura abaixo:

<img src="./Figuras/cloop.png" width="30%" height="30%"/>

Esse sistema é estável como pode ser visto no diagrama de Bode de malha aberta entretanto as margens de estabilidade são bem pequenas. A resposta a degrau unitário enfatiza esse aspecto pelo comportamento bastante oscilatório.

<img src="./Figuras/bodeleadinicial1.png" width="40%" height="40%"/>

<img src="./Figuras/respostadegrauiniciallead1.png" width="40%" height="40%"/>

Para esse sistema projete um controlador avanço de fase que atenda às seguintes especificações:
- Constante de erro de velocidade $K_v=10$,
- Margem de Ganho $G_m \ge 10$dB,
- Margem de Fase $P_m \ge 30^o$.

### Passo 1: Ajuste da constante de erro de velocidade estático

Seja $G_1(s)=KG(s)$
Determinar o valor de $K$ que atende o requisito da constante de erro estático de velocidade $K_v$:
$$
K_v=\lim_{s\rightarrow 0}sG_c(s)G(s) =
$$

$$
K = 
$$


In [ ]:
K=
print("K = ",K)

### Passo 2:

Utilizando o valor de $K$ obtido no passo 1 obtenha o diagrama de Bode para $G_1(j\omega)$. 

verifique no diagrama de Bode a Margem de Ganho $G_m$, a frequência de crossover de ganho $\omega_{cg}$, a Margem de Fase $P_m$ do sistema compensado $G_c(s)G(s)$, a frequência de crossover de fase $\omega_{cp}$ e preencha a tabela abaixo:

|$G_m$ | $\omega_{cg}$ | $Pm$ | $\omega_{cp}$ |
|:--|:--|:--|:--|
| x | x | x | x |

**Obs: utilize o script abaixo.**

In [ ]:
# inline para graficos sem interacao
# qt para grafico em janela separada com interacao (melhora visualizacao)
%matplotlib inline
#%matplotlib qt
import numpy as np
import control.matlab as co
import matplotlib.pyplot as plt
s=co.tf('s')
#
# Sistema
G = 4/(s*((5/2)*s+1)*((1/6)*s+1))
G1 = K*G
##import numpy as np
#import control.matlab as co
#import matplotlib.pyplot as plt

# Utiliza a funcao bode mas sem display grafico
# Aproveita informacoes de modulo, fase e frequencia angular
# Geracao da escala da frequencia angular rad/s
omega = np.logspace(0,2,num=100,base=10)
mag,phase,omega = co.bode(G1,dB=True,Plot=False)
# Converte para dB e graus
magdB = 20*np.log10(mag)
phase_deg = phase*180.0/np.pi
# Utiliza a funcao margin() que calcula
# As margens de estabilidade com suas respetivas freq. de crossover
Gm,Pm,Wcg,Wcp = co.margin(G1)
GmdB = 20*np.log10(Gm)
##Plot Gain and Phase
f, (ax1,ax2) = plt.subplots(2,1)
ax1.semilogx(omega,magdB)
ax1.grid(which="both")
ax1.set_xlabel('Frequency (rad/s)')
ax1.set_ylabel('Magnitude (dB)')
ax2.semilogx(omega,phase_deg)
ax2.grid(which="both")
ax2.set_xlabel('Frequency (rad/s)')
ax2.set_ylabel('Phase (deg)')
ax1.set_title('Gm = '+str(np.round(GmdB,2))+' dB (at '+str(np.round(Wcg,2))+' rad/s), Pm = '+str(np.round(Pm,2))+' deg (at '+str(np.round(Wcp,2))+' rad/s)')
###Plot the zero dB line
ax1.plot(omega,0*omega,'k--',linewidth=2)
###Plot the -180 deg lin
ax2.plot(omega,-180+0*omega,'k--',linewidth=2)
##Plot the vertical line from -180 to 0 at Wcg
ax2.plot([Wcg,Wcg],[-180,0],'r--',linewidth=2)
##Plot the vertical line from -180+Pm to 0 at Wcp
ax2.plot([Wcp,Wcp],[-180+Pm,0],'g--',linewidth=2)
##Plot the vertical line from min(magdB) to 0-GmdB at Wcg
ax1.plot([Wcg,Wcg],[np.min(magdB),0-GmdB],'r--',linewidth=2)
##Plot the vertical line from min(magdB) to 0db at Wcp
ax1.plot([Wcp,Wcp],[np.min(magdB),0],'g--',linewidth=2)


### Passo 3:

Determinar o ângulo de fase $\Phi_m$ que deve ser adicionado para atender à especificação de projeto.

Calcula-se a diferença entre a atual margem de fase $Pm_a$ e a margem de fase mínima especificada pelo projeto $Pm_c$.
**Obs: Usualmente adiciona-se $5^o$ à esse valor já que o ponto de crossover de fase $\omega_{cp}$ se desloca para direita. Se for desejado uma margem de fase maior que o mínimo especificado o valor do ângulo de fase $\Phi_m$ deve ser aumentado.**

$$
\Phi_m = x
$$


### Passo 4:

Dado o ângulo de fase máximo que se deseja acrescentar $\Phi_m$, o parâmetro $\alpha$ pode ser calculado através da seguinte equação:
$$
\sin \Phi_m = \frac{1-\alpha}{1+\alpha},
$$
ou ainda:
$$
\alpha = \frac{1-\sin \Phi_m}{1+\sin \Phi_m} = x
$$

**Obs: utilize o script abaixo.**

In [ ]:
import numpy as np
phim =          # insira o valor do angulo aqui, unidade em graus
# funcao seno so admite angulo em rad
senophim = np.sin(np.radians(phim))
alpha = (1-senophim)/(1+senophim)
print("alpha = ",round(alpha,2)) # Arredonda para 2 casas decimais

### Continuação passo 4:

Determinar a frequência angular $\omega_m$ que corresponde ao ponto onde o controlador tem o maior avanço de fase $\Phi_m$.

Sabemos que a seguinte equação pode ser utilizada:
$$
\omega_m = \frac{1}{\sqrt{\alpha}T},
$$
no entanto não sabemos ainda o valor de $T$. Obs: o zero do controlador é definido por $s=-1/T$ e o pólo $s=-1/\alpha T$.

$$
\left| \frac{1+j\omega T}{1+j\alpha T} \right|_{\omega=\omega_m}=\frac{1}{\sqrt{\alpha}}
$$
ou em dB:
$$
20 \log \left( \frac{1}{\sqrt{\alpha}} \right) = x
$$
**Obs: utilize o script abaixo**

Procura-se agora no diagrama de Bode do módulo de $G_1(j\omega)$ a frequência angular $\omega$ onde o módulo é igual a:
$$
|G_1(j\omega)|= -20 \log \left( \frac{1}{\sqrt{\alpha}} \right)
$$

Essa frequência que corresponde ao maior avanço de fase do controlador é também a nova frequência de crossover:
$$
\omega_c = x
$$

In [ ]:
print("20 log(1/sqrt(alpha)) =",round(20*np.log10(1/np.sqrt(alpha)),2))

### Passo 5: Cálculo das frequências de canto

Sabendo o valor de $\omega_m$ e $\alpha$ podemos o valor das frequências de canto (pólo e zero) do controlador através das seguintes equações:

$$
\frac{1}{T}=\sqrt{\alpha}\omega_c =
$$

$$
\frac{1}{\alpha T}=\frac{\omega_c}{\sqrt{\alpha}} =
$$

**Obs: utilize o script abaixo:**

In [ ]:
# a variavel alpha deve estar definida na celula acima
# coloque aqui o valor escolhido
omegac = 
T=1/round(np.sqrt(alpha)*omegac,2)
print("1/T = ",1/T)
print("1/(alpha T) = ",round(omegac/np.sqrt(alpha),2))

### Passo 6: Cálculo de $K_c$:
$$
K_c=\frac{K}{\alpha}
$$

Podemos escrever agora:
$$
G_c(s) = K_c\alpha \frac{Ts+1}{\alpha Ts+1}=
$$

In [ ]:
Kc = K/alpha
print("Kc = ", round(K/alpha,2))
Gc= Kc * alpha * ((T*s+1)/(alpha*T*s+1))
print("Gc = ",Gc)

### Passo 7: Verificação do projeto

a) Domínio da frequência:

Para observar o resultado no domínio da frequência colocaremos em um único gráfico o Diagrama de Bode de 3 sistemas:
1. $G_1(s)=KG(s)$: a margem de fase é inicialmente medida nesse sistema,
2. $G_c(s)/K$: o ganho de compensação na frequência de crossover é medido nesse sistema,
3. $Gc(s)G(s)$: sistema compensado.

**Obs: utilize o script abaixo**

verifique no diagrama de Bode a Margem de Ganho $G_m$, a frequência de crossover de ganho $\omega_{cg}$, a Margem de Fase $P_m$ do sistema compensado $G_c(s)G(s)$, a frequência de crossover de fase $\omega_{cp}$ e preencha a tabela abaixo:

|$G_m$ | $\omega_{cg}$ | $Pm$ | $\omega_{cp}$ |
|:--|:--|:--|:--|
| x | x | x | x |

In [ ]:
# inline para graficos sem interacao
# qt para grafico em janela separada com interacao (melhora visualizacao)
%matplotlib inline
#%matplotlib qt
import numpy as np
import control.matlab as co
import matplotlib.pyplot as plt
s=co.tf('s')
#
# Sistema
# G1 ja deve estar definido nas celulas acima
#
# Controlador Gck = Gc/K
# Gc e K ja devem estar definidos
#
Gck = Gc/K
#
# Malha aberta
openloop = Gc*G
#
# Geracao da escala da frequencia angular rad/s
# 10**fi ---> 10**ff
fi=-2
ff=2
omega = np.logspace(fi,ff,num=100,base=10)
# Utiliza a funcao bode mas sem display grafico
# Aproveita informacoes de modulo, fase
# a escala omega e' imposta 
mag,phase,omega1 = co.bode(openloop,omega,dB=True,Plot=False)
# Converte para dB e graus
magdB = 20*np.log10(mag)
phase_deg = phase*180.0/np.pi
#
# Utiliza a funcao margin() que calcula
# as margens de estabilidade com suas respetivas freq. de crossover
# de Gc(s)G(s)=openloop
Gm,Pm,Wcg,Wcp = co.margin(openloop)
# Converte para dB
GmdB = 20*np.log10(Gm)
#
# Diagrama de Bode de G1
#
magg1,phaseg1,omega1 = co.bode(G1,omega,dB=True,Plot=False)
# Converte para dB e graus
magg1dB = 20*np.log10(magg1)
phaseg1_deg = phaseg1*180.0/np.pi
#
# Diagrama de Bode de Gc(s)/K
#
maggck,phasegck,omega1 = co.bode(Gck,omega,dB=True,Plot=False)
# Converte para dB e graus
maggckdB = 20*np.log10(maggck)
phasegck_deg = phasegck*180.0/np.pi
#
# Coloca tudo no grafico
# G1(s), Gc(s)/K, Gc(s)G(s)
#
f, (ax1,ax2) = plt.subplots(2,1)
ax1.semilogx(omega,magdB,omega,magg1dB,omega,maggckdB)
ax1.grid(which="both")
ax1.set_xlabel('Frequency (rad/s)')
ax1.set_ylabel('Magnitude (dB)')
ax2.semilogx(omega,phase_deg,omega,phaseg1_deg,omega,phasegck_deg)
ax2.grid(which="both")
ax2.set_xlabel('Frequency (rad/s)')
ax2.set_ylabel('Phase (deg)')
ax1.set_title('Gm = '+str(np.round(GmdB,2))+' dB (at '+str(np.round(Wcg,2))+' rad/s), Pm = '+str(np.round(Pm,2))+' deg (at '+str(np.round(Wcp,2))+' rad/s)')
#
# linhas auxiliares no grafico
#
###Plot the zero dB line
ax1.plot(omega,0*omega,'k--',linewidth=2)
###Plot the -180 deg lin
ax2.plot(omega,-180+0*omega,'k--',linewidth=2)
##Plot the vertical line from -180 to 0 at Wcg
ax2.plot([Wcg,Wcg],[-180,0],'r--',linewidth=2)
##Plot the vertical line from -180+Pm to 0 at Wcp
ax2.plot([Wcp,Wcp],[-180+Pm,0],'g--',linewidth=2)
##Plot the vertical line from min(magdB) to 0-GmdB at Wcg
ax1.plot([Wcg,Wcg],[np.min(magdB),0-GmdB],'r--',linewidth=2)
##Plot the vertical line from min(magdB) to 0db at Wcp
ax1.plot([Wcp,Wcp],[np.min(magdB),0],'g--',linewidth=2)

### Continuação passo 7:

**b) Domínio do tempo**

Para o domínio do tempo será comparado o sistema inicial $G(s)$ e o sistema compensado
$G_c(s)G(s)$ ambos em malha fechada:

$$
\frac{Y(s)}{R(s)}=\frac{G(s)}{1+G(s)}
$$
e
$$
\frac{Y(s)}{R(s)}=\frac{Gc(s)G(s)}{1+G_c(s)G(s)}
$$

b-1) Resposta a degrau unitário:

**Obs: utilize o script abaixo**

Anote na tabela a seguir as grandezas tempo de subida $t_r$, tempo de acomodação $t_s$, máximo sobresinal $M_p$ e valor máximo de esforço de controle $\max u(t)$ (Estimar o valor pelo gráfico).

|Sistema | $t_r$ | $t_s$ | $M_p$ | $$\max u(t)$$ |
|:--|:--|:--|:--|:--|
| $$\frac{G(s)}{1+G(s)}$$ |x|x|x|x|
| $$\frac{G_c(s)G(s)}{1+G_c(s)G(s)}$$ |x|x|x|x|

- Compare as duas soluções.

**[Resposta]**


In [ ]:
# inline para graficos sem interacao
# qt para grafico em janela separada com interacao
%matplotlib inline
#%matplotlib qt
#
# R(s)  E(s |------|  Y(s)
#---->(+)---| G(s) |------->
#    _ ^    |------|    |
#      |-----------------
#
# R(s)  E(s)|-------|  |------|  Y(s)
#---->(+)---| Gc(s) |--| G(s) |------->
#    _ ^    |-------|  |------|    |
#      |--------------------------- 
#
cloop1 = co.feedback(G,1)
print('\nPOLOS E ZEROS DE MALHA FECHADA')
print('-------------')
print('Polos e zeros cloop1')
print('Polos = ',co.pole(cloop1))
print('Zeros = ',co.zero(cloop1))
print('COEF. DE AMORTECIMENTO E FREQ. NATURAL')
print('_____Polos____________zeta_______omegan')
co.damp(cloop1)
cloop2 = co.feedback(Gc*G,1)
print('\nPOLOS E ZEROS DE MALHA FECHADA')
print('-------------')
print('Polos e zeros cloop2')
print('Polos = ',co.pole(cloop2))
print('Zeros = ',co.zero(cloop2))
print('COEF. DE AMORTECIMENTO E FREQ. NATURAL')
print('_____Polos____________zeta_______omegan')
co.damp(cloop2)
# base de tempo t
plt.figure(1)
t=[x*0.05 for x in range(0,200)]
y1, t = co.step(cloop1,t)
y2, t = co.step(cloop2,t)
plt.plot(t,y1,t,y2)
plt.title('Resposta a degrau do sistema em malha fechada')
plt.xlabel('tempo (s)')
plt.ylabel('y')
plt.grid()
# Calcula as caracteristicas da resposta transitoria
#  stepinfo(sys, T=None, SettlingTimeThreshold=0.02, RiseTimeLimits=(0.1,0.9))
#  S: a dictionary containing:
#        RiseTime: Time from 10% to 90% of the steady-state value.
#        SettlingTime: Time to enter inside a default error of 2%
#        SettlingMin: Minimum value after RiseTime
#        SettlingMax: Maximum value after RiseTime
#        Overshoot: Percentage of the Peak relative to steady value
#        Undershoot: Percentage of undershoot
#        Peak: Absolute peak value
#        PeakTime: time of the Peak
#        SteadyStateValue: Steady-state value
S1 = co.stepinfo(cloop1)
print('-------------')
print('CARACTERISTICAS DA RESPOSTA TRANSITORIA DO SISTEMA cloop1')
print('tempo de subida tr = ','%.2f' % S1['RiseTime'],'seg')
print('tempo de acomodacao ts = ','%.2f' % S1['SettlingTime'],'seg')
print('maximo sobresinal Mp = ',S1['Overshoot'])
print('valor de pico ymax = ','%.2f' % S1['Peak'])
print('instante de pico tp = ','%.2f' % S1['PeakTime'],'seg')
print('valor de regime estacionario yss = ','%.2f' % S1['SteadyStateValue'])
S2 = co.stepinfo(cloop2)
print('-------------')
print('CARACTERISTICAS DA RESPOSTA TRANSITORIA DO SISTEMA cloop2')
print('tempo de subida tr = ','%.2f' % S2['RiseTime'],'seg')
print('tempo de acomodacao ts = ','%.2f' % S2['SettlingTime'],'seg')
print('maximo sobresinal Mp = ',S2['Overshoot'])
print('valor de pico ymax = ','%.2f' % S2['Peak'])
print('instante de pico tp = ','%.2f' % S2['PeakTime'],'seg')
print('valor de regime estacionario yss = ','%.2f' % S2['SteadyStateValue'])
# Esforco de controle
esforco1 = co.feedback(1,G)
esforco2 = co.feedback(Gc,G)
vp1, t = co.step(esforco1,t)
vp2, t = co.step(esforco2,t)
plt.figure(2)
plt.plot(t,vp1,t,vp2)
plt.title('Esforco de controle')
plt.xlabel('tempo (s)')
plt.ylabel('u(t)')
plt.grid()

b-2) Resposta a rampa:

**Obs: utilize o script abaixo**

- O erro estático $e_{ss}$ não é nulo. Justifique.

**[Resposta]**


In [ ]:
# inline para graficos sem interacao
# qt para grafico em janela separada com interacao
%matplotlib inline
#%matplotlib qt
# Calculo da resposta a rampa
t=[x*0.05 for x in range(0,150)]
u = t.copy()
y1, tempo, x1 = co.lsim(cloop1,u,t)
y2, tempo, x2 = co.lsim(cloop2,u,t)
plt.plot(t,u,t,y1,t,y2)
plt.title('Resposta a rampa do sistema em malha fechada')
plt.xlabel('tempo (s)')
plt.ylabel('y')
plt.grid()

### Exercício 3: Projeto com controlador atraso de fase

O controlador por atraso de fase pode ser escrito através da seguinte equação:
$$
G_c(s) = K_c \beta \frac{Ts+1}{\beta Ts+1} = K_c \frac{s+\frac{1}{T}}{s+\frac{1}{\beta T}}
$$

Um diagrama de Bode para $K_c=1$ e $\beta=10$ é ilustrado abaixo:

<img src="./Figuras/lagcontrol.png" width="50%" height="50%"/>

Seja o sistema definido por:
$$
G(s) = \frac{1}{s(s+1)(s+0.5)},
$$

O sistema de controle em malha fechada correspondente é ilustrado na figura abaixo:

<img src="./Figuras/cloop.png" width="30%" height="30%"/>

Esse sistema é instável como pode ser visto no diagrama de Bode de malha aberta e pela resposta a degrau unitário.

<img src="./Figuras/bodelaginicial.png" width="40%" height="40%"/>

<img src="./Figuras/respostadegrauiniciallag.png" width="40%" height="40%"/>

Para esse sistema projete um controlador atraso de fase que atenda às seguintes especificações:
- Constante de erro de velocidade $K_v=20$
- Margem de Ganho $G_m \ge 10$ dB
- Margem de Fase $P_m \ge 40$


### Passo 1: Ajuste da constante de erro de velocidade estático

Seja $G_1(s)=KG(s)$
Determinar o valor de $K$ que atende o requisito da constante de erro estático de velocidade $K_v$:
$$
K_v=\lim_{s\rightarrow 0}sG_c(s)G(s) = x
$$

$$
K = x
$$

**Obs: insira o valor de $K$ obtido na célula abaixo**

In [ ]:
K=

### Passo 2:

Utilizando o valor de $K$ obtido no passo 1 obtenha o diagrama de Bode para $G_1(j\omega)$.

Verifique no diagrama de Bode a Margem de Ganho $G_m$, a frequência de crossover de ganho $\omega_{cg}$, a Margem de Fase $P_m$, a frequência de crossover de fase $\omega_{cp}$ e preencha a tabela abaixo:

|$G_m$ | $\omega_{cg}$ | $Pm$ | $\omega_{cp}$ |
|:--|:--|:--|:--|
| x | x | x | x |

Se a margem de fase não for satisfatória procure a frequência angular onde a fase é $-180^o + P_{mc} + (5^o a 12^o)$.
**Onde $P_{mc}$ é a especificação de projeto para a margem de Fase e o ângulo adicional tenta compensar o atraso de fase introduzido pelo controlador.**

$$
\omega_{cp} = x
$$

**Obs: utilize o script abaixo.**

**Para analisar utilize a opção qt para melhor visualização , ao final mantenha a célula ativa com gráfico inline**

In [ ]:
# inline para graficos sem interacao
# qt para grafico em janela separada com interacao (melhora visualizacao)
%matplotlib inline
#%matplotlib qt
import numpy as np
import control.matlab as co
import matplotlib.pyplot as plt
s=co.tf('s')
#
# Sistema
G  = 1/(s*(s+1)*(s+0.5))
G1 = K*G
print("G1 = ",G1)
#
# Utiliza a funcao bode mas sem display grafico
# Aproveita informacoes de modulo, fase e frequencia angular
# Geracao da escala da frequencia angular rad/s
omega = np.logspace(0,2,num=100,base=10)
mag,phase,omega = co.bode(G1,dB=True,Plot=False)
# Converte para dB e graus
magdB = 20*np.log10(mag)
phase_deg = phase*180.0/np.pi
# Utiliza a funcao margin() que calcula
# As margens de estabilidade com suas respetivas freq. de crossover
Gm,Pm,Wcg,Wcp = co.margin(G1)
GmdB = 20*np.log10(Gm)
##Plot Gain and Phase
f, (ax1,ax2) = plt.subplots(2,1)
ax1.semilogx(omega,magdB)
ax1.grid(which="both")
ax1.set_xlabel('Frequency (rad/s)')
ax1.set_ylabel('Magnitude (dB)')
ax2.semilogx(omega,phase_deg)
ax2.grid(which="both")
ax2.set_xlabel('Frequency (rad/s)')
ax2.set_ylabel('Phase (deg)')
ax1.set_title('Gm = '+str(np.round(GmdB,2))+' dB (at '+str(np.round(Wcg,2))+' rad/s), Pm = '+str(np.round(Pm,2))+' deg (at '+str(np.round(Wcp,2))+' rad/s)')
###Plot the zero dB line
ax1.plot(omega,0*omega,'k--',linewidth=2)
###Plot the -180 deg lin
ax2.plot(omega,-180+0*omega,'k--',linewidth=2)
##Plot the vertical line from -180 to 0 at Wcg
ax2.plot([Wcg,Wcg],[-180,0],'r--',linewidth=2)
##Plot the vertical line from -180+Pm to 0 at Wcp
ax2.plot([Wcp,Wcp],[-180+Pm,0],'g--',linewidth=2)
##Plot the vertical line from min(magdB) to 0-GmdB at Wcg
ax1.plot([Wcg,Wcg],[np.min(magdB),0-GmdB],'r--',linewidth=2)
##Plot the vertical line from min(magdB) to 0db at Wcp
ax1.plot([Wcp,Wcp],[np.min(magdB),0],'g--',linewidth=2)


### Passo 3: Escolha da frequência de canto $\omega_z=1/T$

Para evitar os efeitos do atraso de fase do controlador a posição da frequência de canto correspondente ao zero do controlador é escolhido de uma oitava (metade) a uma década (dez vezes menor) abaixo da frequência de crossover
$\omega_{cp}$ escolhida.

Indique abaixo a frequência escolhida:
$$
\omega_z=1/T=x
$$

**Obs: utilize o script abaixo**

In [ ]:
omegaz = 
print("omegaz = ",omegaz)
T = 1/omegaz
print("T = ",T)

### Passo 4: Determinação de $\beta$

Determinar a atenuação do controlador para que na frequência de crossover escolhida $\omega_{cp}$ o sistema compensado $G_c(s)G(s)$ tenha ganho $0$ dB.

Ou seja, $|G_c(j\omega_{cp})/K|$ deve ser o mesmo que $|G_1(j\omega_{cp})|$ com sinal  oposto.
Essa atenuação pode ser estimada como $-20\log \beta$.

-$|G_1(j\omega_{cp})|=|G_c(j\omega_{cp})/K|= -20\log \beta$


Indique abaixo

$$
\beta =
$$
e
$$
\omega = 1/(\beta T)=
$$

**Obs: utilize o script abaixo**

In [ ]:
# x é igual a -|G1(jwcp)|
x=
beta = 10**(-x/20)
print("beta = ",beta)

### Passo 5: Constante $K_c$

O valor de $K_c$ pode ser determinado através de:

$$
K_c = \frac{K}{\beta}
$$

**Obs: utilize o script abaixo**

In [ ]:
Kc=K/beta
print("Kc = ",Kc)

### Passo 6: Controlador $Gc(s)$

Podemos agora definir o controlador
$$
G_c(s) = K_c \beta \frac{Ts+1}{\beta Ts+1} =K \frac{Ts+1}{\beta Ts+1}
$$
e
$$
G_c(s)/K = \frac{Ts+1}{\beta Ts+1}
$$

**Obs: utilize o script abaixo**

In [ ]:
Gc = K*(T*s+1)/(beta*T*s+1)
print("Gc = ",Gc)
Gck = Gc/K
print("Gck = ",Gck)

### Passo 6: Verificação

a) Domínio da frequência:

Para observar o resultado no domínio da frequência colocaremos em um único gráfico o Diagrama de Bode de 3 sistemas:
1. $G_1(s)=KG(s)$: a margem de fase é inicialmente medida nesse sistema,
2. $G_c(s)/K$: o ganho de compensação na frequência de crossover é medido nesse sistema,
3. $Gc(s)G(s)$: sistema compensado.

verifique no diagrama de Bode a Margem de Ganho $G_m$, a frequência de crossover de ganho $\omega_{cg}$, a Margem de Fase $P_m$ do sistema compensado $G_c(s)G(s)$, a frequência de crossover de fase $\omega_{cp}$ e preencha a tabela abaixo:

|$G_m$ | $\omega_{cg}$ | $Pm$ | $\omega_{cp}$ |
|:--|:--|:--|:--|
| x | x | x | x |

**Obs: utilize o script abaixo**

In [ ]:
#%matplotlib qt
%matplotlib inline
import numpy as np
import control.matlab as co
import matplotlib.pyplot as plt
s=co.tf('s')
#
# Sistema
# G1 ja deve estar definido nas celulas acima
#
# Controlador Gck = Gc/K
#
# Malha aberta
openloop = Gc*G
#
# Geracao da escala da frequencia angular rad/s
# 10**fi ---> 10**ff
fi=-4
ff=1
omega = np.logspace(fi,ff,num=100,base=10)
# Utiliza a funcao bode mas sem display grafico
# Aproveita informacoes de modulo, fase
# a escala omega e' imposta 
mag,phase,omega1 = co.bode(openloop,omega,dB=True,Plot=False)
# Converte para dB e graus
magdB = 20*np.log10(mag)
phase_deg = phase*180.0/np.pi
#
# Utiliza a funcao margin() que calcula
# as margens de estabilidade com suas respetivas freq. de crossover
# de Gc(s)G(s)=openloop
Gm,Pm,Wcg,Wcp = co.margin(openloop)
# Converte para dB
GmdB = 20*np.log10(Gm)
#
# Diagrama de Bode de G1
#
magg1,phaseg1,omega1 = co.bode(G1,omega,dB=True,Plot=False)
# Converte para dB e graus
magg1dB = 20*np.log10(magg1)
phaseg1_deg = phaseg1*180.0/np.pi
#
# Diagrama de Bode de Gc(s)/K
#
maggck,phasegck,omega1 = co.bode(Gck,omega,dB=True,Plot=False)
# Converte para dB e graus
maggckdB = 20*np.log10(maggck)
phasegck_deg = phasegck*180.0/np.pi
#
# Coloca tudo no grafico
# G1(s), Gc(s)/K, Gc(s)G(s)
#
f, (ax1,ax2) = plt.subplots(2,1)
ax1.semilogx(omega,magdB,omega,magg1dB,omega,maggckdB)
ax1.grid(which="both")
ax1.set_xlabel('Frequency (rad/s)')
ax1.set_ylabel('Magnitude (dB)')
ax2.semilogx(omega,phase_deg,omega,phaseg1_deg,omega,phasegck_deg)
ax2.grid(which="both")
ax2.set_xlabel('Frequency (rad/s)')
ax2.set_ylabel('Phase (deg)')
ax1.set_title('Gm = '+str(np.round(GmdB,2))+' dB (at '+str(np.round(Wcg,2))+' rad/s), Pm = '+str(np.round(Pm,2))+' deg (at '+str(np.round(Wcp,2))+' rad/s)')
#
# linhas auxiliares no grafico
#
###Plot the zero dB line
ax1.plot(omega,0*omega,'k--',linewidth=2)
###Plot the -180 deg lin
ax2.plot(omega,-180+0*omega,'k--',linewidth=2)
##Plot the vertical line from -180 to 0 at Wcg
ax2.plot([Wcg,Wcg],[-180,0],'r--',linewidth=2)
##Plot the vertical line from -180+Pm to 0 at Wcp
ax2.plot([Wcp,Wcp],[-180+Pm,0],'g--',linewidth=2)
##Plot the vertical line from min(magdB) to 0-GmdB at Wcg
ax1.plot([Wcg,Wcg],[np.min(magdB),0-GmdB],'r--',linewidth=2)
##Plot the vertical line from min(magdB) to 0db at Wcp
ax1.plot([Wcp,Wcp],[np.min(magdB),0],'g--',linewidth=2)

### Continuação passo 6:

**b) Domínio do tempo**

Para o domínio do tempo será comparado o sistema inicial $G(s)$ e o sistema compensado
$G_c(s)G(s)$ ambos em malha fechada:

$$
\frac{Y(s)}{R(s)}=\frac{G(s)}{1+G(s)}
$$
e
$$
\frac{Y(s)}{R(s)}=\frac{Gc(s)G(s)}{1+G_c(s)G(s)}
$$

b-1) Resposta a degrau unitário:

**Obs: utilize o script abaixo**

Anote na tabela a seguir as grandezas tempo de subida $t_r$, tempo de acomodação $t_s$, máximo sobresinal $M_p$ e valor máximo de esforço de controle $\max u(t)$ (Estimar o valor pelo gráfico) para o sistema compensado. Para o sistema não compensado não é possível definir tais grandezas porque o sistema é instável.

|Sistema | $t_r$ | $t_s$ | $M_p$ | $$\max u(t)$$ |
|:--|:--|:--|:--|:--|
| $$\frac{G_c(s)G(s)}{1+G_c(s)G(s)}$$ |x|x|x|x|

- Compare as duas soluções.

**[Resposta]**


In [ ]:
# inline para graficos sem interacao
# qt para grafico em janela separada com interacao
%matplotlib inline
#%matplotlib qt
#
# R(s)  E(s |------|  Y(s)
#---->(+)---| G(s) |------->
#    _ ^    |------|    |
#      |-----------------
#
# R(s)  E(s)|-------|  |------|  Y(s)
#---->(+)---| Gc(s) |--| G(s) |------->
#    _ ^    |-------|  |------|    |
#      |--------------------------- 
#
cloop1 = co.feedback(G,1)
print('\nPOLOS E ZEROS DE MALHA FECHADA')
print('-------------')
print('Polos e zeros cloop1')
print('Polos = ',co.pole(cloop1))
print('Zeros = ',co.zero(cloop1))
print('COEF. DE AMORTECIMENTO E FREQ. NATURAL')
print('_____Polos____________zeta_______omegan')
co.damp(cloop1)
cloop2 = co.feedback(Gc*G,1)
print('\nPOLOS E ZEROS DE MALHA FECHADA')
print('-------------')
print('Polos e zeros cloop2')
print('Polos = ',co.pole(cloop2))
print('Zeros = ',co.zero(cloop2))
print('COEF. DE AMORTECIMENTO E FREQ. NATURAL')
print('_____Polos____________zeta_______omegan')
co.damp(cloop2)
# base de tempo t
plt.figure(1)
t=[x*0.1 for x in range(0,500)]
y1, t = co.step(cloop1,t)
y2, t = co.step(cloop2,t)
plt.plot(t,y1,t,y2)
plt.title('Resposta a degrau do sistema em malha fechada')
plt.xlabel('tempo (s)')
plt.ylabel('y')
plt.grid()
# Calcula as caracteristicas da resposta transitoria
#  stepinfo(sys, T=None, SettlingTimeThreshold=0.02, RiseTimeLimits=(0.1,0.9))
#  S: a dictionary containing:
#        RiseTime: Time from 10% to 90% of the steady-state value.
#        SettlingTime: Time to enter inside a default error of 2%
#        SettlingMin: Minimum value after RiseTime
#        SettlingMax: Maximum value after RiseTime
#        Overshoot: Percentage of the Peak relative to steady value
#        Undershoot: Percentage of undershoot
#        Peak: Absolute peak value
#        PeakTime: time of the Peak
#        SteadyStateValue: Steady-state value
# Esforco de controle
esforco1 = co.feedback(1,G)
esforco2 = co.feedback(Gc,G)
vp1, t = co.step(esforco1,t)
vp2, t = co.step(esforco2,t)
plt.figure(2)
plt.plot(t,vp1,t,vp2)
plt.title('Esforco de controle')
plt.xlabel('tempo (s)')
plt.ylabel('u(t)')
plt.grid()

b-2) Resposta a rampa:

**Obs: utilize o script abaixo**

- O erro estático $e_{ss}$ não é nulo. Justifique.

**[Resposta]**


In [ ]:
# inline para graficos sem interacao
# qt para grafico em janela separada com interacao
%matplotlib inline
#%matplotlib qt
# Calculo da resposta a rampa
#t=[x*0.05 for x in range(0,50)]
u = t.copy()
y1, tempo, x1 = co.lsim(cloop1,u,t)
y2, tempo, x2 = co.lsim(cloop2,u,t)
plt.plot(t,u,t,y1,t,y2)
plt.title('Resposta a rampa do sistema em malha fechada')
plt.xlabel('tempo (s)')
plt.ylabel('y')
plt.grid()

### 5. Discussões

### 6. Conclusões